# Notebook 6 — Train, Tune, Evaluate

## Objective

In this notebook, we train and tune machine learning models to predict whether an order will be delivered on time or late.

The notebook will:
- Establish a simple baseline for comparison.
- Train a machine learning model using the training data.
- Tune the model using the validation split.
- Use evaluation metrics appropriate for the imbalanced target variable.
- Evaluate the final selected model on the test set only once, at the end.
- Save the trained model and a results summary as artifacts.

## 1. Setup

We begin by importing the libraries required for data loading, model training, evaluation, and artifact management.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

from xgboost import XGBClassifier

## 2. Load the Final Feature Table

The final feature table generated in Notebook 5 contains the engineered and preprocessed features for the training, validation, and test splits.

The `split` column identifies which rows belong to each dataset split. In this notebook, we reuse these predefined splits rather than creating a new split.

In [2]:
feature_table_path = "../Artifacts/final_feature_table.csv"

final_feature_table = pd.read_csv(feature_table_path)

print("Feature table shape:", final_feature_table.shape)
print("\nSplit distribution:")
print(final_feature_table["split"].value_counts())

Feature table shape: (96476, 62)

Split distribution:
split
train         67533
test          14472
validation    14471
Name: count, dtype: int64


## 3. Prepare the Target Variable

The target variable is `late_flag`, where 0 represents an on-time delivery and 1 represents a late delivery.

The target is loaded from the labeled dataset and aligned with the predefined train, validation, and test splits used in Notebook 5.

In [3]:
labeled_table = pd.read_csv("../Artifacts/labeled_table.csv")

print(labeled_table.shape)
print(labeled_table.columns.tolist())

(96476, 14)
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'number_of_items', 'total_price', 'total_freight', 'number_of_payments', 'total_payment_value', 'delivery_label']


In [4]:
print(
    "Same number of rows:",
    len(final_feature_table) == len(labeled_table)
)

print(
    "Same order IDs:",
    final_feature_table.index.equals(labeled_table.index)
)

Same number of rows: True
Same order IDs: True


## 4. Create Train, Validation, and Test Sets

The predefined splits from Notebook 5 are used to separate the final feature table into training, validation, and test sets.

The target variable is aligned with the same row order as the feature table, ensuring that each feature row corresponds to the correct delivery label.

In [5]:
# Add the target and split information together
model_data = final_feature_table.copy()
model_data["delivery_label"] = labeled_table["delivery_label"].values

# Convert the target to binary
model_data["late_flag"] = (
    model_data["delivery_label"] == "late"
).astype(int)

# Load the feature list created in Notebook 5
with open("../Artifacts/feature_list.json", "r") as f:
    feature_list = json.load(f)

print("Number of features:", len(feature_list))

# Separate the splits
train_data = model_data[model_data["split"] == "train"].copy()
validation_data = model_data[model_data["split"] == "validation"].copy()
test_data = model_data[model_data["split"] == "test"].copy()

# Separate features and target
X_train = train_data[feature_list]
y_train = train_data["late_flag"]

X_validation = validation_data[feature_list]
y_validation = validation_data["late_flag"]

X_test = test_data[feature_list]
y_test = test_data["late_flag"]

print("X_train:", X_train.shape)
print("X_validation:", X_validation.shape)
print("X_test:", X_test.shape)

print("\nTarget distribution:")
print("Train:")
print(y_train.value_counts())
print("\nValidation:")
print(y_validation.value_counts())
print("\nTest:")
print(y_test.value_counts())

Number of features: 61
X_train: (67533, 61)
X_validation: (14471, 61)
X_test: (14472, 61)

Target distribution:
Train:
late_flag
0    62978
1     4555
Name: count, dtype: int64

Validation:
late_flag
0    13456
1     1015
Name: count, dtype: int64

Test:
late_flag
0    13507
1      965
Name: count, dtype: int64


## 5. Baseline

Because the target variable is imbalanced, we establish a simple majority-class baseline before training a machine learning model.

The baseline predicts every order as `on_time` (0). Its performance provides a reference point that the trained model should improve upon, particularly for detecting late deliveries.

In [6]:
# Majority-class baseline: predict every order as on-time
y_validation_baseline = np.zeros(len(y_validation), dtype=int)

print("Baseline classification report:")
print(
    classification_report(
        y_validation,
        y_validation_baseline,
        target_names=["on_time", "late"],
        zero_division=0
    )
)

print("Baseline confusion matrix:")
print(confusion_matrix(y_validation, y_validation_baseline))

Baseline classification report:


              precision    recall  f1-score   support

     on_time       0.93      1.00      0.96     13456
        late       0.00      0.00      0.00      1015

    accuracy                           0.93     14471
   macro avg       0.46      0.50      0.48     14471
weighted avg       0.86      0.93      0.90     14471

Baseline confusion matrix:
[[13456     0]
 [ 1015     0]]


## 6. Evaluation Metric

The target variable is imbalanced, with substantially fewer late deliveries than on-time deliveries.

Therefore, accuracy alone is not sufficient for model selection. The primary evaluation metric in this notebook is the F1-score for the `late` class, as it balances precision and recall for the minority class.

Recall for the `late` class will also be monitored because identifying late deliveries is an important business objective.

## 7. Train the Initial Model

We train an initial XGBoost classifier using the training split as the starting point for model development.

At this stage, no tuning or threshold adjustment is performed. The validation split will be used to assess the initial model and guide subsequent tuning, while the test split remains untouched until the final evaluation.

In [7]:
xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train,
    y_train
)

print("Initial XGBoost model trained successfully.")

Initial XGBoost model trained successfully.


## 8. Evaluate the Initial Model on the Validation Set

The initial model is evaluated on the validation split to establish its performance before tuning.

The test set is not used at this stage. We focus on precision, recall, and F1-score for the `late` class because the target variable is imbalanced.

In [8]:
# Predict on the validation set
y_validation_pred = xgb_model.predict(X_validation)

print("Initial XGBoost - Validation Performance:")
print(
    classification_report(
        y_validation,
        y_validation_pred,
        target_names=["on_time", "late"],
        zero_division=0
    )
)

print("Confusion Matrix:")
print(confusion_matrix(y_validation, y_validation_pred))

Initial XGBoost - Validation Performance:
              precision    recall  f1-score   support

     on_time       0.93      1.00      0.96     13456
        late       0.00      0.00      0.00      1015

    accuracy                           0.93     14471
   macro avg       0.46      0.50      0.48     14471
weighted avg       0.86      0.93      0.90     14471

Confusion Matrix:
[[13456     0]
 [ 1015     0]]


## 9. Handle Class Imbalance

The initial model predicted almost all observations as on-time because the target variable is imbalanced.

To give the minority `late` class more influence during training, we use XGBoost's `scale_pos_weight` parameter. The weight is calculated from the training split only as the ratio of on-time observations to late observations.

In [9]:
# Calculate class weight from the training data only
n_on_time = (y_train == 0).sum()
n_late = (y_train == 1).sum()

scale_pos_weight = n_on_time / n_late

print("On-time:", n_on_time)
print("Late:", n_late)
print("scale_pos_weight:", scale_pos_weight)

On-time: 62978
Late: 4555
scale_pos_weight: 13.826125137211855


## 10. Train an Imbalance-Aware XGBoost Model

The initial XGBoost model failed to identify the minority `late` class.

To address this, we train a second XGBoost model using the calculated `scale_pos_weight`. All other initial model parameters are kept unchanged so that the effect of class weighting can be evaluated on the validation split.

In [10]:
xgb_weighted = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_weighted.fit(
    X_train,
    y_train
)

print("Imbalance-aware XGBoost model trained successfully.")

Imbalance-aware XGBoost model trained successfully.


## 11. Evaluate the Imbalance-Aware Model

The imbalance-aware XGBoost model is evaluated on the validation split.

Its performance is compared with the initial model, with particular attention to the F1-score and recall of the `late` class. The test set remains untouched.

In [11]:
# Predict on the validation set
y_validation_pred_weighted = xgb_weighted.predict(X_validation)

print("Imbalance-Aware XGBoost - Validation Performance:")
print(
    classification_report(
        y_validation,
        y_validation_pred_weighted,
        target_names=["on_time", "late"],
        zero_division=0
    )
)

print("Confusion Matrix:")
print(confusion_matrix(
    y_validation,
    y_validation_pred_weighted
))

Imbalance-Aware XGBoost - Validation Performance:
              precision    recall  f1-score   support

     on_time       0.93      0.71      0.81     13456
        late       0.07      0.29      0.11      1015

    accuracy                           0.68     14471
   macro avg       0.50      0.50      0.46     14471
weighted avg       0.87      0.68      0.76     14471

Confusion Matrix:
[[9554 3902]
 [ 716  299]]


## 12. Tune the Class Weight

The initial imbalance-aware model improved detection of late deliveries, but its precision for the `late` class remained low.

We therefore tune the `scale_pos_weight` parameter using the training and validation splits. The F1-score of the `late` class is used as the primary metric for selecting the best configuration.

The test set is not used during tuning.

In [12]:
scale_pos_weights = [5, 8, 10, 13.826, 16, 20]

tuning_results = []

for weight in scale_pos_weights:

    model = XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        colsample_bytree=0.8,
        scale_pos_weight=weight,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_validation)

    precision = precision_score(
        y_validation,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_validation,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_validation,
        y_pred,
        zero_division=0
    )

    tuning_results.append({
        "scale_pos_weight": weight,
        "precision_late": precision,
        "recall_late": recall,
        "f1_late": f1
    })

tuning_results_df = pd.DataFrame(tuning_results)

print(tuning_results_df)

   scale_pos_weight  precision_late  recall_late   f1_late
0             5.000        0.000000     0.000000  0.000000
1             8.000        0.076923     0.004926  0.009259
2            10.000        0.076271     0.026601  0.039445
3            13.826        0.071174     0.294581  0.114647
4            16.000        0.068037     0.525123  0.120466
5            20.000        0.069068     0.731034  0.126212


## 13. Select the Best Configuration

The best configuration is selected based on the highest F1-score for the `late` class on the validation set.

Among the tested values, the configuration with the highest validation F1-score is selected for the final model.

In [13]:
best_result = tuning_results_df.loc[
    tuning_results_df["f1_late"].idxmax()
]

best_scale_pos_weight = best_result["scale_pos_weight"]

print("Best scale_pos_weight:", best_scale_pos_weight)
print("Best validation F1 (late):", best_result["f1_late"])

Best scale_pos_weight: 20.0
Best validation F1 (late): 0.12621194080625958


## 14. Train the Final Model

The final XGBoost model is trained using the selected configuration from validation-based tuning.

The selected `scale_pos_weight` is 20, which achieved the highest F1-score for the `late` class among the tested configurations.

The test set remains untouched and will only be used for the final evaluation.

In [14]:
final_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    colsample_bytree=0.8,
    scale_pos_weight=best_scale_pos_weight,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

final_model.fit(
    X_train,
    y_train
)

print("Final XGBoost model trained successfully.")

Final XGBoost model trained successfully.


## 15. Final Evaluation on the Test Set

After selecting the final model using the training and validation splits, we evaluate it once on the held-out test set.

The test set is used only to estimate the final generalization performance of the selected model and is not used for further model selection or tuning.

In [15]:
# Final predictions on the test set
y_test_pred = final_model.predict(X_test)

print("Final XGBoost - Test Performance:")
print(
    classification_report(
        y_test,
        y_test_pred,
        target_names=["on_time", "late"],
        zero_division=0
    )
)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

Final XGBoost - Test Performance:
              precision    recall  f1-score   support

     on_time       0.93      0.35      0.51     13507
        late       0.07      0.65      0.12       965

    accuracy                           0.37     14472
   macro avg       0.50      0.50      0.32     14472
weighted avg       0.88      0.37      0.48     14472

Confusion Matrix:
[[4735 8772]
 [ 341  624]]


## 16. Collect Evaluation Results

The baseline, tuning, and final test results are collected into structured objects so they can be saved as a reusable results summary artifact.

In [16]:
# Collect final test metrics for the late class
final_test_precision = precision_score(
    y_test,
    y_test_pred,
    zero_division=0
)

final_test_recall = recall_score(
    y_test,
    y_test_pred,
    zero_division=0
)

final_test_f1 = f1_score(
    y_test,
    y_test_pred,
    zero_division=0
)

print("Final Test Metrics - Late Class")
print("Precision:", final_test_precision)
print("Recall:", final_test_recall)
print("F1:", final_test_f1)

Final Test Metrics - Late Class
Precision: 0.06641123882503193
Recall: 0.6466321243523316
F1: 0.12045169385194479


## 17. Save Results Summary

The evaluation results are saved as a JSON artifact for reproducibility and future reference.

The summary records the baseline performance, class-weight tuning results, selected model configuration, and final test performance.

In [17]:
import os

results_summary = {
    "primary_metric": "f1_late",
    "secondary_metric": "recall_late",

    "data_splits": {
        "train": len(y_train),
        "validation": len(y_validation),
        "test": len(y_test)
    },

    "baseline": {
        "validation_accuracy": float(
            accuracy_score(y_validation, y_validation_baseline)
        ),
        "late_precision": float(
            precision_score(
                y_validation,
                y_validation_baseline,
                zero_division=0
            )
        ),
        "late_recall": float(
            recall_score(
                y_validation,
                y_validation_baseline,
                zero_division=0
            )
        ),
        "late_f1": float(
            f1_score(
                y_validation,
                y_validation_baseline,
                zero_division=0
            )
        )
    },

    "tuning": tuning_results,

    "selected_configuration": {
        "model": "XGBClassifier",
        "n_estimators": 300,
        "learning_rate": 0.05,
        "max_depth": 5,
        "colsample_bytree": 0.8,
        "scale_pos_weight": float(best_scale_pos_weight)
    },

    "final_test": {
        "precision_late": float(final_test_precision),
        "recall_late": float(final_test_recall),
        "f1_late": float(final_test_f1),
        "accuracy": float(
            accuracy_score(y_test, y_test_pred)
        )
    }
}

os.makedirs("../Artifacts", exist_ok=True)

with open("../Artifacts/results_summary.json", "w") as f:
    json.dump(results_summary, f, indent=4)

print("Results summary saved successfully.")

Results summary saved successfully.


## 18. Save the Final Model

The selected final XGBoost model is saved as a reusable artifact.

This model uses the configuration selected through validation-based tuning and can be loaded later for inference without retraining.

In [18]:

os.makedirs("../Artifacts/models", exist_ok=True)

model_path = "../Artifacts/models/xgboost_final_model.joblib"

joblib.dump(
    final_model,
    model_path
)

print("Final model saved successfully.")
print("Path:", model_path)

Final model saved successfully.
Path: ../Artifacts/models/xgboost_final_model.joblib


## 19. Conclusion

In this notebook, we established a majority-class baseline, trained an initial XGBoost classifier, and addressed the class imbalance using `scale_pos_weight`.

The model configuration was tuned using the validation split, with F1-score for the `late` class selected as the primary metric. The best tested configuration used a `scale_pos_weight` of 20 and achieved a validation F1-score of approximately 0.126 for late deliveries.

The final model was then evaluated once on the held-out test set. It achieved a precision of approximately 6.6%, recall of approximately 64.7%, and F1-score of approximately 12.0% for the `late` class.

The results show that the model can identify a substantial proportion of late deliveries, but its low precision indicates a high number of false positive predictions. This trade-off should be considered when using the model for practical decision-making.

The final XGBoost model and the results summary were saved as reusable artifacts for future inference and deployment.